# RetinaNet for NFL Helmets
RetinaNet is a model that is a competitor to the object detection models. Its main highlights is that it utilizes a Feature Pyramid Network (FPN) in order to make much better predictions. Compared to the state-of-the-art YOLO models, RetinaNet is slightly slower and less performative.

So then why RetinaNet? This architecture excels at detecting small and partially obscured objects. Therefore, it can be perfect for predicting NFL Helmets on a live feed of the game, especially since many helmets can get obscured by neighbouring players. Additionally, RetinaNet is built right into Torchvision, whereas YOLO needs its own run configurations and is extremely difficult to adapt to other frameworks, such as PyTorch Lightning.

We will use the data from the Kaggle competition of detecting NFL helmets and build up a model using the still images, and then apply it to the video.

## Import Packages

In [25]:
from typing import Optional

import numpy as np
import polars
import os
import glob
import datetime as dt
import shutil
import itertools
import copy
import json

import plotly.express as px
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from tqdm.notebook import tqdm
import cv2

import pytorch_lightning as pl
import torch
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.models.detection import RetinaNet, RetinaNet_ResNet50_FPN_V2_Weights

RANDOM_SEED = 1729
TEST_SIZE = 0.2

DATA_DIR = os.path.join('../data', 'nfl-health-and-safety-helmet-assignment')
DATASET_DIR = os.path.join(DATA_DIR, 'nfl_helmet_image_dataset')

## Training on Still Images
Before we jump into the videos, we will train on the provided ~10000 images. But before we use the images directly, we need to place them in the right format. This includes structuring our directories correctly, with separate train and validation folders. Within each folder, we have two sub-folders for the images and labels. We will create this structure for each view we have.

Originally, the images are located in the top-level directory with `./data/nfl-health-and-safety-helmet-assignment`. We will create another folder here and establish the format.

```text
dataset/
├── train/
│   ├── images
│       ├── images1.jpg
│       ├── images2.jpg
│       └── ...other images
│   └── labels
│       ├── images1.txt
│       ├── images2.txt
│       └── ...other label files
├── valid/
│   ├── images
│   └── labels
└── test/
    ├── images
    └── labels
```

### Putting Images into Structure

In [2]:
ENDZONE_DIR = os.path.join(DATASET_DIR, 'Endzone')
SIDELINE_DIR = os.path.join(DATASET_DIR, 'Sideline')
os.makedirs(ENDZONE_DIR, exist_ok=True)
os.makedirs(SIDELINE_DIR, exist_ok=True)

In [3]:
ENDZONE_DIR = os.path.join(DATASET_DIR, 'Endzone')
SIDELINE_DIR = os.path.join(DATASET_DIR, 'Sideline')
os.makedirs(ENDZONE_DIR, exist_ok=True)
os.makedirs(SIDELINE_DIR, exist_ok=True)
# Create the directories. We will just have a train and test datasets.
# First find the split...
# Only try moving if the directory is NOT EMPTY
if len(glob.glob(os.path.join(ENDZONE_DIR, '**/*.jpg'), recursive=True)) == 0:
    for view, directory in zip(['Endzone', 'Sideline'], [ENDZONE_DIR, SIDELINE_DIR]):
        print(f'Gathering the {view} view...')
        all_og_image_files = glob.glob(os.path.join(DATA_DIR, 'images', f'*{view}*.jpg'))
        # Create a train test split. Make it reproducible...
        train, valid = train_test_split(all_og_image_files, test_size=TEST_SIZE, random_state=RANDOM_SEED)
        print('Training images:', len(train))
        print('Validation images:', len(valid))
        # print('Test images:', len(test))
        # Create the train and test directories inside...
        os.makedirs(os.path.join(directory, 'train', 'images'), exist_ok=True)
        os.makedirs(os.path.join(directory, 'valid', 'images'), exist_ok=True)
        # os.makedirs(os.path.join(directory, 'test', 'images'), exist_ok=True)
        # Move the files to the sub-directory (/train or /test)
        # Put it into the further directory of "images"
        for dirname, dataset in zip(['train', 'valid'], [train, valid]):
            for file in tqdm(dataset, desc=f'Moving {dirname} files'):
                shutil.move(file, os.path.join(directory, dirname, 'images', os.path.basename(file)))
        print()


### Creating Annotations
Now that the images are in the right structure, the next step is to generate the text files that represents the bounding boxes. Each image will have its own corresponding text file that lists all the objects present in the image. The text file will have the same filename. Each line will have the format `class_id x_min y_min x_max y_max`.

In other words we save the top left and bottom right corners of all the boxes. Because our images are 1280 by 720, and they will be resized to 640 by 640 for the model, we will save **normalized** coordinates of the boxes.

For now let's convert our box information in `image_labels.csv` to the text files. For the purposes of this project, we just have one kind of annotation, the helmet. In our ground truth labeling, we have `Helmet`, `Helmet-Blurred`, `Helmet-Difficult`, and `Helmet-Sideline`. We will consolidate all of these into a single `Helmet` label.

In [4]:
image_labels = polars.read_csv(os.path.join(DATA_DIR, 'image_labels.csv'))
print(image_labels.head())
print(image_labels.shape)

shape: (5, 6)
┌─────────────────────────────────┬────────┬──────┬───────┬─────┬────────┐
│ image                           ┆ label  ┆ left ┆ width ┆ top ┆ height │
│ ---                             ┆ ---    ┆ ---  ┆ ---   ┆ --- ┆ ---    │
│ str                             ┆ str    ┆ i64  ┆ i64   ┆ i64 ┆ i64    │
╞═════════════════════════════════╪════════╪══════╪═══════╪═════╪════════╡
│ 57503_000116_Endzone_frame443.… ┆ Helmet ┆ 1099 ┆ 16    ┆ 456 ┆ 15     │
│ 57503_000116_Endzone_frame443.… ┆ Helmet ┆ 1117 ┆ 15    ┆ 478 ┆ 16     │
│ 57503_000116_Endzone_frame443.… ┆ Helmet ┆ 828  ┆ 16    ┆ 511 ┆ 15     │
│ 57503_000116_Endzone_frame443.… ┆ Helmet ┆ 746  ┆ 16    ┆ 519 ┆ 16     │
│ 57503_000116_Endzone_frame443.… ┆ Helmet ┆ 678  ┆ 17    ┆ 554 ┆ 17     │
└─────────────────────────────────┴────────┴──────┴───────┴─────┴────────┘
(193736, 6)


In [5]:
# Add columns that tell us what view it is, and whether it falls into the train or test set.
structured_files = glob.glob(os.path.join(DATASET_DIR, '**/*.jpg'), recursive=True)
filenames = [os.path.basename(file) for file in structured_files]
dirnames = [os.path.basename(os.path.dirname(os.path.dirname(file))) for file in structured_files]
get_train_test = lambda filename: dirnames[filenames.index(os.path.basename(filename))]
image_labels = image_labels.with_columns(
    polars.when(polars.col('image').str.contains('Endzone'))
        .then(polars.lit('Endzone'))
        .otherwise(polars.lit('Sideline'))
        .alias('view'),
    polars.col('image').map_elements(get_train_test, return_dtype=polars.String).alias('split')
)
print(image_labels.head())

shape: (5, 8)
┌─────────────────────────────────┬────────┬──────┬───────┬─────┬────────┬─────────┬───────┐
│ image                           ┆ label  ┆ left ┆ width ┆ top ┆ height ┆ view    ┆ split │
│ ---                             ┆ ---    ┆ ---  ┆ ---   ┆ --- ┆ ---    ┆ ---     ┆ ---   │
│ str                             ┆ str    ┆ i64  ┆ i64   ┆ i64 ┆ i64    ┆ str     ┆ str   │
╞═════════════════════════════════╪════════╪══════╪═══════╪═════╪════════╪═════════╪═══════╡
│ 57503_000116_Endzone_frame443.… ┆ Helmet ┆ 1099 ┆ 16    ┆ 456 ┆ 15     ┆ Endzone ┆ train │
│ 57503_000116_Endzone_frame443.… ┆ Helmet ┆ 1117 ┆ 15    ┆ 478 ┆ 16     ┆ Endzone ┆ train │
│ 57503_000116_Endzone_frame443.… ┆ Helmet ┆ 828  ┆ 16    ┆ 511 ┆ 15     ┆ Endzone ┆ train │
│ 57503_000116_Endzone_frame443.… ┆ Helmet ┆ 746  ┆ 16    ┆ 519 ┆ 16     ┆ Endzone ┆ train │
│ 57503_000116_Endzone_frame443.… ┆ Helmet ┆ 678  ┆ 17    ┆ 554 ┆ 17     ┆ Endzone ┆ train │
└─────────────────────────────────┴────────┴──────┴─────

We will iterate through each image, get the rows that have its bounding boxes, and create a file and write it. If we have orders of magnitude more files, then we can parallelize this process if need be.

**For starters, we will treat all helmets the same, including those of the sideline.**

In [26]:
for view, split in itertools.product(['Endzone', 'Sideline'], ['train', 'valid']):

    # Filter the dataframe according to the view and split,
    # and SORT according to the image file name.
    # This will come in handy later.
    dataset_view_split = image_labels.filter(
        polars.col('view').eq(view),
        polars.col('split').eq(split)
    )
    # Create a labels directory under the split directory
    label_dir = os.path.join(DATASET_DIR, view, split, 'labels')
    os.makedirs(label_dir, exist_ok=True)
    image_names = dataset_view_split['image'].unique().to_list()
    # For each file, write the labels to the
    for image in tqdm(image_names, desc=f'Creating labels for {view} ({split})'):
        # Filter the dataset_view_split to get the bounding boxes for this image...
        # Also add a class_id column with just zeroes.
        this_image_boxes = dataset_view_split.filter(polars.col('image').eq(image)).with_columns(class_id=0)
        # Read image object to get the width and height
        # image_obj = Image.open(os.path.join(DATASET_DIR, view, split, 'images', image))
        image_obj = cv2.imread(os.path.join(DATASET_DIR, view, split, 'images', image))
        image_width, image_height = image_obj.shape[1], image_obj.shape[0]
        # Extract the coordinates in the correct column, and convert to numpy
        this_image_boxes = this_image_boxes[['class_id', 'left', 'top', 'width', 'height']].to_numpy().astype(object)
        # We need x_min, y_min, x_max, y_max.
        # The first two columns are already x_min and y_min, so let us add
        # the width and height of the box to get x_max and y_max
        this_image_boxes[:, 3] += this_image_boxes[:, 1]
        this_image_boxes[:, 4] += this_image_boxes[:, 2]
        # Normalize the x-coordinates by the width, and the y-coordinates by the height
        this_image_boxes[:, [1, 3]] /= image_width
        this_image_boxes[:, [2, 4]] /= image_height
        # Use np.savetxt to output the file directly.
        # The first column is the ID, and so it needs to be written as an integer.
        # The other 4 columns are floats, and let's write it out to 6 decimal places.
        np.savetxt(os.path.join(label_dir, f"{image[:image.rindex('.')]}.txt"), this_image_boxes,
                   delimiter=',', fmt='%d,%.6f,%.6f,%.6f,%.6f')


Creating labels for Endzone (train):   0%|          | 0/3991 [00:00<?, ?it/s]

Creating labels for Endzone (valid):   0%|          | 0/998 [00:00<?, ?it/s]

Creating labels for Sideline (train):   0%|          | 0/3966 [00:00<?, ?it/s]

Creating labels for Sideline (valid):   0%|          | 0/992 [00:00<?, ?it/s]

## Creating Dataset and DataModule Class
The label files are now all in the right format, so it's time to set up the `LightningDataModule` and `LightningModule` that will house our module and full model.

The `DataModule` will load the image and its labels, and do some transforms on the data, and return them. The train-test split will be done during setup of this module.

The `Module` will load the RetinaNet model and provide calls to the model, and also load the data loaders that are necessary.

In [27]:
image_files = sorted(glob.glob(os.path.join(DATASET_DIR, 'Endzone', 'train', 'images', '*.jpg')))
label_files = sorted(glob.glob(os.path.join(DATASET_DIR, 'Endzone', 'train', 'labels', '*.txt')))

In [28]:
image_files[:10]

['../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/images/57502_000480_Endzone_frame0495.jpg',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/images/57502_002958_Endzone_frame0584.jpg',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/images/57502_003762_Endzone_frame1071.jpg',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/images/57502_004004_Endzone_frame0436.jpg',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/images/57503_000116_Endzone_frame443.jpg',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/images/57503_000969_Endzone_frame0630.jpg',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/images/57503_001581_Endzone_frame327.jpg',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_data

In [29]:
label_files[:10]

['../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/labels/57502_000480_Endzone_frame0495.txt',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/labels/57502_002958_Endzone_frame0584.txt',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/labels/57502_003762_Endzone_frame1071.txt',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/labels/57502_004004_Endzone_frame0436.txt',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/labels/57503_000116_Endzone_frame443.txt',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/labels/57503_000969_Endzone_frame0630.txt',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/labels/57503_001581_Endzone_frame327.txt',
 '../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_data

In [36]:
# This is a basic Dataset class (no PyTorch Lightning specialities here).
# This class will be the one that is actually doing the transforming
# and returning of the image and its bounding box labels.
class NFLHelmetDataset(Dataset):
    def __init__(self, data_folder, transform, width, height):
        # Under data_folder, there should be a sub-directory
        # of "images" and "labels"
        self.data_folder = data_folder
        self.transform = transform
        self.width = width
        self.height = height
        # The filenames are identical other than the extension,
        # so sorting both should pair each other perfectly.
        self.image_files = sorted(glob.glob(os.path.join(data_folder, 'images', '*.jpg')))
        self.label_files = sorted(glob.glob(os.path.join(data_folder, 'labels', '*.txt')))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # We need to read both the image and its corresponding bounding box text file
        image_file = self.image_files[idx]
        label_file = self.label_files[idx]
        print(image_file, label_file)
        # Read the image, and convert to RGB and divide by 255 to convert to floats
        image = cv2.imread(image_file)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32)

        # 2) Resize image (to the model's expected size)
        image_resized = cv2.resize(image, (self.width, self.height))
        image_resized /= 255.0  # Scale pixel values to [0, 1]

        # Next, we need to read the bounding boxes from the text file.
        # The box is standardized and is of the format class_id, x_min, y_min, x_max, y_max
        # However, since we need to transform the image, and take the boxes along for the ride,
        # we need to both put the boxes and labels into an array (for albumentations)
        # and also create a dictionary to act as the "target".
        labels = np.loadtxt(label_file, dtype=np.float32, delimiter=',')
        print(labels)

        # We are going to convert the normalized coordinates into their absolute
        # coordinates using the width and height. Additionally, we need to separate
        # out the bounding box labels, and convert both the boxes and labels into tensors.
        labels, boxes = labels[:, 0].astype(np.int64), labels[:, 1:]
        # Convert to absolute coordinates.
        # Remember it's x_min, y_min, x_max, y_max
        boxes[:, [0, 2]] *= 640
        boxes[:, [1, 3]] *= 640
        # If the max coordinate is somehow larger than the min coordinate, then correct it.
        boxes[boxes[:, 2] <= boxes[:, 0], 2] = boxes[boxes[:, 2] <= boxes[:, 0], 0] + 1
        boxes[boxes[:, 3] <= boxes[:, 1], 3] = boxes[boxes[:, 3] <= boxes[:, 1], 1] + 1
        # Clip everything to within the image size (min coordinate needs to be size - 1)
        boxes[:, [0, 1]] = np.clip(boxes[:, [0, 1]], 0, 640 - 1)
        boxes[:, [2, 3]] = np.clip(boxes[:, [2, 3]], 0, 640)

        labels = torch.from_numpy(labels)
        boxes = torch.from_numpy(boxes)

        # Create the dictionary that will hold the "target", which will have the
        # bounding boxes and its labels, as well as some other info
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        is_crowd = torch.zeros((len(boxes),), dtype=torch.int64)
        image_id = torch.tensor([idx])

        target = {"boxes": boxes, "labels": labels, "area": area, "iscrowd": is_crowd, "image_id": image_id}

        if self.transform is not None:
            # To apply albumentations transformations, the boxes and labels
            # need to be normal numpy arrays
            labels_list = labels.cpu().numpy()
            boxes_list = boxes.cpu().numpy()

            transformed_image = self.transform(
                image=image_obj,
                bboxes=boxes_list,
                labels=labels_list
            )

            # The transformed image and corresponding transformed boxes are in attributes...
            new_boxes = transformed_image['bboxes'].astype(np.float32)
            new_labels = transformed_image['labels'].astype(np.int64)
            # Convert to tensors and update the dictionary
            new_boxes = torch.from_numpy(new_boxes)
            new_labels = torch.from_numpy(new_labels)

            target['boxes'] = new_boxes
            target['labels'] = new_labels


        return image_resized, target

class NFLHelmetLightningDataModule(pl.LightningDataModule):
    def __init__(self, view: str, image_size: int = 640, data_dir: str = DATASET_DIR, downsample_n: int = -1,
                 validation_split: float = 0.2, batch_size: int = 32, num_workers: int = 1, pin_memory: bool = False):
        super().__init__()
        # Save all the hyperparaameters
        self.save_hyperparameters()
        self.train_dataset: Optional[Dataset] = None
        self.val_dataset: Optional[Dataset] = None

        # Get the directory based on the camera view we are training for
        self.view_dir = os.path.join(data_dir, view)

        # Below are the transformations that will be applied to the training and validation images.
        # Validation generally only has resizing transforms.
        # Below transforms can effectively "increase" the number of camera
        # angles in our dataset, as well as create "new" team jerseys through
        # the color jitter.
        self.train_transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=45),
            A.Blur(blur_limit=3, p=0.2),
            A.MotionBlur(blur_limit=3, p=0.1),
            A.MedianBlur(blur_limit=3, p=0.1),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2, p=0.3),
            A.RandomScale(scale_limit=0.2, p=0.3),
            ToTensorV2(p=1.0)
        ])
        # For validation, all we do is convert to a tensor
        self.val_transform = A.Compose([ToTensorV2(p=1.0)])

    def prepare_data(self) -> None:
        # Normally, this is where we would put the data preprocessing i.e. the moving
        # of folders and creation of the label txts. But we have done that already above.
        pass

    def setup(self, stage: Optional[str] = None) -> None:
        pass



In [79]:
transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=45),
        A.Blur(blur_limit=3, p=0.2),
        A.MotionBlur(blur_limit=3, p=0.1),
        A.MedianBlur(blur_limit=3, p=0.1),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2, p=0.3),
        A.RandomScale(scale_limit=0.2, p=0.3),
        ToTensorV2(p=1.0)
    ],
    bbox_params={"format": "pascal_voc", "label_fields": ["labels"]})
dataset = NFLHelmetDataset(os.path.join(ENDZONE_DIR, 'train'), transform, 640, 640)

In [38]:
dataset[0]

../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/images/57502_000480_Endzone_frame0495.jpg ../data/nfl-health-and-safety-helmet-assignment/nfl_helmet_image_dataset/Endzone/train/labels/57502_000480_Endzone_frame0495.txt
[[0.       0.314844 0.411111 0.338281 0.456944]
 [0.       0.328906 0.465278 0.35     0.522222]
 [0.       0.395313 0.488889 0.416406 0.5375  ]
 [0.       0.445312 0.493056 0.464844 0.544444]
 [0.       0.507812 0.544444 0.527344 0.590278]
 [0.       0.482812 0.631944 0.505469 0.666667]
 [0.       0.560937 0.497222 0.582812 0.545833]
 [0.       0.608594 0.495833 0.628906 0.547222]
 [0.       0.6625   0.473611 0.684375 0.526389]
 [0.       0.703906 0.422222 0.725    0.473611]
 [0.       0.500781 0.301389 0.519531 0.343056]
 [0.       0.611719 0.180556 0.629687 0.220833]
 [0.       0.839844 0.555556 0.863281 0.579167]
 [0.       0.784375 0.558333 0.798438 0.569444]
 [0.       0.739844 0.544444 0.757812 0.559722]
 [0.       0.142969 0.

In [80]:
labels = np.loadtxt(os.path.join(ENDZONE_DIR, 'train', 'labels', '57502_000480_Endzone_frame0495.txt'),
                    dtype=np.float32, delimiter=',')
labels, boxes = labels[:, 0].astype(np.int64), labels[:, 1:]
# Convert to absolute coordinates.
# Remember it's x_min, y_min, x_max, y_max
boxes[:, [0, 2]] *= 640
boxes[:, [1, 3]] *= 640
# If the max coordinate is somehow larger than the min coordinate, then correct it.
boxes[boxes[:, 2] <= boxes[:, 0], 2] = boxes[boxes[:, 2] <= boxes[:, 0], 0] + 1
boxes[boxes[:, 3] <= boxes[:, 1], 3] = boxes[boxes[:, 3] <= boxes[:, 1], 1] + 1
# Clip everything to within the image size (min coordinate needs to be size - 1)
boxes[:, [0, 1]] = np.clip(boxes[:, [0, 1]], 0, 640 - 1)
boxes[:, [2, 3]] = np.clip(boxes[:, [2, 3]], 0, 640)

labels = torch.from_numpy(labels)
boxes = torch.from_numpy(boxes)

print(boxes)


tensor([[201.5002, 263.1110, 216.4998, 292.4442],
        [210.4998, 297.7779, 224.0000, 334.2221],
        [253.0003, 312.8890, 266.4998, 344.0000],
        [284.9997, 315.5558, 297.5002, 348.4442],
        [324.9997, 348.4442, 337.5002, 377.7780],
        [308.9997, 404.4442, 323.5002, 426.6669],
        [358.9997, 318.2221, 372.9997, 349.3331],
        [389.5002, 317.3331, 402.4998, 350.2221],
        [424.0000, 303.1110, 438.0000, 336.8890],
        [450.4998, 270.2221, 464.0000, 303.1110],
        [320.4998, 192.8890, 332.4998, 219.5558],
        [391.5002, 115.5558, 402.9997, 141.3331],
        [537.5001, 355.5558, 552.4999, 370.6669],
        [502.0000, 357.3331, 511.0003, 364.4442],
        [473.5002, 348.4442, 484.9997, 358.2221],
        [ 91.5002, 344.0000, 103.5002, 354.6669],
        [186.9997, 345.7779, 201.0003, 360.0000]])


In [81]:
image_obj = cv2.imread(os.path.join(ENDZONE_DIR, 'train', 'images', '57502_000480_Endzone_frame0495.jpg')).astype(np.float32)
image_obj = cv2.resize(image_obj, (640, 640))
fig = px.imshow(cv2.cvtColor(image_obj, cv2.COLOR_BGR2RGB))
fig.add_shape(
    type='rect',
    x0=201.5002,
    y0=263.1110,
    x1=216.4998,
    y1=292.4442,
    line={'color': 'magenta'}
)
fig.show()

In [82]:
image_obj /= 255.0
labels_list = labels.cpu().numpy()
boxes_list = boxes.cpu().numpy()

transformed_image = transform(
    image=image_obj,
    bboxes=boxes_list,
    labels=labels_list
)

In [83]:
new_image = transformed_image['image'].cpu().numpy() * 255
new_image = np.clip(np.round(new_image), 0, 255).astype(np.uint8)
new_image

array([[[ 73,  81,  76, ...,  57,  71,  73],
        [ 74,  79,  76, ...,  55,  71,  73],
        [ 71,  77,  78, ...,  51,  70,  73],
        ...,
        [ 72,  76,  75, ..., 171, 157, 153],
        [ 67,  64,  62, ..., 136, 128, 123],
        [ 65,  65,  61, ...,  82,  80,  78]],

       [[134, 154, 148, ..., 132, 132, 128],
        [136, 152, 147, ..., 130, 131, 128],
        [133, 149, 148, ..., 125, 130, 127],
        ...,
        [123, 141, 146, ..., 204, 180, 172],
        [119, 129, 133, ..., 184, 166, 157],
        [117, 130, 132, ..., 142, 129, 125]],

       [[115, 130, 122, ..., 121, 108, 105],
        [116, 127, 121, ..., 119, 108, 105],
        [112, 124, 121, ..., 115, 108, 106],
        ...,
        [104, 120, 123, ..., 198, 170, 162],
        [100, 108, 111, ..., 184, 162, 152],
        [ 98, 109, 110, ..., 143, 127, 120]]],
      shape=(3, 521, 521), dtype=uint8)

In [84]:
new_image.shape

(3, 521, 521)

In [91]:
new_boxes = transformed_image['bboxes']
new_labels = transformed_image['labels']

print(new_boxes)

[[344.75558054 214.18881401 356.96626961 238.06781816]
 [338.64998758 242.40983781 349.63997418 272.07765287]
 [304.05245554 254.71118909 315.04191422 280.03751242]
 [278.81628203 256.88217562 288.99246699 283.6553368 ]
 [246.25378203 283.6553368  256.42993593 307.53485334]
 [257.6506384  329.24282438 269.45495456 347.33349895]
 [217.35494214 259.05266529 228.75182956 284.37898862]
 [193.33996797 258.32899794 203.92252582 285.1026715 ]
 [164.44063121 246.75131401 175.83748758 274.24867046]
 [143.27498758 219.97765598 154.26497418 246.75131401]
 [250.32434297 157.02367356 260.09309918 178.73217252]
 [192.93306714  94.06967562 202.29439461 115.0540026 ]
 [ 71.23056191 289.44467562  83.44131309 301.74601138]
 [105.01379544 290.89148241 112.34061879 296.68032438]
 [126.17993593 283.6553368  135.5412634  291.61516529]
 [436.74441946 280.03751242 446.51314461 288.72099274]
 [357.37317044 281.48484713 368.77058578 293.0625    ]]


In [92]:
fig = px.imshow(new_image.transpose((1, 2, 0)))
fig.add_shape(
    type='rect',
    x0=344.7555,
    y0=214.1888,
    x1=356.96626,
    y1=238.0678,
    line={'color': 'magenta'}
)
fig.show()